# 06 - Exercise 2: collect MLS data (ASA)
Exercise 2: central midfielder shortlist for Nashville SC

Brief (assumed): defensive midfielder, under 26, salary ≤ $1M, market value ≤ €7M, other MLS club, 2026 season. 60% ball-winning, 40% progression.

Source: [American Soccer Analysis](https://app.americansocceranalysis.com) API. Downloaded 12 Nov 2026. ASA can revise past numbers.

1. Players: name, birth date, nationality
2. Teams
3. Goals added (g+) by action type
4. xPass: passing vs expected, pass distance
5. xG, xA, key passes
6. Salaries: guaranteed compensation (MLSPA)
7. Combine
8. Missing values check
9. Save
10. Team possession
11. Second half of 2025

## Setup

In [ ]:
from pathlib import Path

import requests
import pandas as pd

pd.set_option("display.max_columns", 50)

SEASON = "2026"
TODAY = pd.Timestamp("2026-11-11")      # used for age ("under 26 today")

API_URL = "https://app.americansocceranalysis.com/api/v1/mls"

PROCESSED_FOLDER = Path("../../data/processed")

### API helper

In [ ]:
def get_asa(endpoint, params=None):
    url = f"{API_URL}/{endpoint}"
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    return pd.DataFrame(response.json())

## 1 - Players

In [ ]:
players_raw = get_asa("players")
print(players_raw.shape)
players_raw.columns.tolist()

In [ ]:
players = players_raw[["player_id", "player_name", "birth_date", "nationality", "primary_general_position"]].copy()
players["birth_date"] = pd.to_datetime(players["birth_date"])

# age today
players["age"] = (TODAY - players["birth_date"]).dt.days / 365.25
players.head()

## 2 - Teams

In [ ]:
teams_raw = get_asa("teams")
teams_raw.columns.tolist()

In [ ]:
teams = teams_raw[["team_id", "team_name", "team_abbreviation"]]

# team_id -> team name
team_name_by_id = dict(zip(teams["team_id"], teams["team_name"]))

NASHVILLE_ID = teams[teams["team_name"].str.contains("Nashville")]["team_id"].iloc[0]
print("Nashville team_id:", NASHVILLE_ID)

## 3 - Goals added (g+)
Value of each action by its effect on scoring and conceding. Action types:
- Interrupting: tackles, interceptions, blocks, recoveries (ball-winning)
- Passing: passes (progression)
- Receiving: getting on the ball in good spots (progression)
- Dribbling: carries, take-ons (ball losses check)
- Shooting: shots (not used)
- Fouling: fouls won (+) and committed (−)

In [ ]:
goals_added_raw = get_asa("players/goals-added", params={"season_name": SEASON})
print(goals_added_raw.shape)
goals_added_raw.head(3)

Goals added comes nested, one entry per action type:

In [ ]:
pd.DataFrame(goals_added_raw["data"].iloc[0])

One row per player, one column per action type

In [ ]:
def flatten_goals_added(raw_table):
    flat_rows = []

    for index, row in raw_table.iterrows():
        new_row = {"player_id": row["player_id"]}

        for action in row["data"]:
            name = action["action_type"].lower()                  # e.g. "interrupting"
            new_row[f"gplus_{name}"] = action["goals_added_raw"]
            new_row[f"actions_{name}"] = action["count_actions"]

        flat_rows.append(new_row)

    return pd.DataFrame(flat_rows)


goals_added_wide = flatten_goals_added(goals_added_raw)
goals_added_wide.head(3)

In [ ]:
# keep the basic info columns too (team, position, minutes)
basic_info = goals_added_raw[["player_id", "team_id", "general_position", "minutes_played"]]
goals_added = basic_info.merge(goals_added_wide, on="player_id")
goals_added.shape

## 4 - xPass
- Passes completed over expected, per 100 passes
- Average forward distance of passes

In [ ]:
xpass_raw = get_asa("players/xpass", params={"season_name": SEASON})
xpass_raw.columns.tolist()

In [ ]:
xpass = xpass_raw[["player_id", "attempted_passes", "pass_completion_percentage", "xpass_completion_percentage",
                   "passes_completed_over_expected", "passes_completed_over_expected_p100",
                   "avg_distance_yds", "avg_vertical_distance_yds", "share_team_touches", "count_games"]]
xpass.head(3)

## 5 - xG and xA

In [ ]:
xgoals_raw = get_asa("players/xgoals", params={"season_name": SEASON})
xgoals = xgoals_raw[["player_id", "goals", "xgoals", "primary_assists", "xassists", "key_passes"]]
xgoals.head(3)

## 6 - Salaries (MLSPA)
Guaranteed compensation, latest release per player.

In [ ]:
salaries_raw = get_asa("players/salaries", params={"season_name": SEASON})
print(salaries_raw.shape)
print("Releases:", salaries_raw["mlspa_release"].unique())
salaries_raw.head(3)

In [ ]:
salaries = salaries_raw.copy()
salaries["guaranteed_compensation"] = pd.to_numeric(salaries["guaranteed_compensation"])   # stored as text

salaries = salaries.sort_values("mlspa_release")
salaries = salaries.drop_duplicates("player_id", keep="last")      # latest release per player

salaries = salaries[["player_id", "team_id", "guaranteed_compensation", "mlspa_release"]]
salaries = salaries.rename(columns={"team_id": "salary_team_id"})

## 7 - Combine
Left merges onto goals added (players with 2026 minutes).

In [ ]:
mls_2026 = goals_added.merge(players, on="player_id", how="left")
mls_2026 = mls_2026.merge(xpass, on="player_id", how="left")
mls_2026 = mls_2026.merge(xgoals, on="player_id", how="left")
mls_2026 = mls_2026.merge(salaries, on="player_id", how="left")
mls_2026.shape

### Players who changed teams
36 players played for more than one team in 2026.
- Played for Nashville: excluded later
- Current team: team in the latest salary release, otherwise the last team they played for

In [ ]:
def to_list(team_id):
    # make every team_id a list
    if isinstance(team_id, list):
        return team_id
    return [team_id]

mls_2026["team_ids"] = mls_2026["team_id"].apply(to_list)
mls_2026["changed_team"] = mls_2026["team_ids"].apply(len) > 1
print("Players who changed team:", mls_2026["changed_team"].sum())

In [ ]:
def played_for_nashville(team_ids):
    return NASHVILLE_ID in team_ids

def current_team_id(row):
    if pd.notna(row["salary_team_id"]):
        return row["salary_team_id"]
    return row["team_ids"][-1]

mls_2026["played_for_nashville"] = mls_2026["team_ids"].apply(played_for_nashville)
mls_2026["current_team_id"] = mls_2026.apply(current_team_id, axis=1)
mls_2026["current_team"] = mls_2026["current_team_id"].map(team_name_by_id)

mls_2026[mls_2026["changed_team"]][["player_name", "team_ids", "current_team"]].head()

## 8 - Missing values check

In [ ]:
key_columns = ["player_name", "age", "minutes_played", "gplus_interrupting", "avg_vertical_distance_yds",
               "xassists", "guaranteed_compensation"]
mls_2026[key_columns].isna().sum()

Missing values are players with few minutes. Quick count of players fitting the brief (filtering in notebook 07):

In [ ]:
is_midfielder = mls_2026["general_position"].isin(["CM", "DM"])
enough_minutes = mls_2026["minutes_played"] >= 1200
under_26 = mls_2026["age"] < 26
not_nashville = ~mls_2026["played_for_nashville"]
salary_ok = mls_2026["guaranteed_compensation"] <= 1_000_000

print("Midfielders (CM/DM):      ", is_midfielder.sum())
print("+ 1,200+ minutes:         ", (is_midfielder & enough_minutes).sum())
print("+ under 26:               ", (is_midfielder & enough_minutes & under_26).sum())
print("+ not Nashville:          ", (is_midfielder & enough_minutes & under_26 & not_nashville).sum())
print("+ salary <= $1M:          ", (is_midfielder & enough_minutes & under_26 & not_nashville & salary_ok).sum())

## 9 - Save
Combined table only (the raw tables have mixed list columns that parquet can't store).

In [ ]:
# list column, saved as text
mls_2026_to_save = mls_2026.drop(columns=["team_id"]).copy()
def join_ids(team_ids):
    return ",".join(team_ids)          # ["a", "b"] -> "a,b"

mls_2026_to_save["team_ids"] = mls_2026_to_save["team_ids"].apply(join_ids)
mls_2026_to_save.to_parquet(PROCESSED_FOLDER / f"asa_players_{SEASON}.parquet")

print("Saved:", PROCESSED_FOLDER / f"asa_players_{SEASON}.parquet", mls_2026_to_save.shape)

## 10 - Team possession
For the possession-adjusted check in notebook 07. Possession ≈ passes for / (for + against).

In [ ]:
team_xpass_raw = get_asa("teams/xpass", params={"season_name": SEASON})

team_possession = team_xpass_raw[["team_id", "attempted_passes_for", "attempted_passes_against"]].copy()
team_possession["possession"] = team_possession["attempted_passes_for"] / (
    team_possession["attempted_passes_for"] + team_possession["attempted_passes_against"])
team_possession["team_name"] = team_possession["team_id"].map(team_name_by_id)

team_possession.to_parquet(PROCESSED_FOLDER / f"asa_team_possession_{SEASON}.parquet")
team_possession.sort_values("possession", ascending=False)[["team_name", "possession"]].round(2).head()

## 11 - Second half of 2025
1 Jul – 31 Dec 2025, goals added and xPass, for the consistency check.

In [ ]:
H2_2025 = {"start_date": "2025-07-01", "end_date": "2025-12-31"}

goals_added_h2_raw = get_asa("players/goals-added", params=H2_2025)
xpass_h2_raw = get_asa("players/xpass", params=H2_2025)
print("Goals added rows:", len(goals_added_h2_raw), "| xPass rows:", len(xpass_h2_raw))
print("Most minutes by one player:", goals_added_h2_raw["minutes_played"].max())

In [ ]:
basic_info_h2 = goals_added_h2_raw[["player_id", "general_position", "minutes_played"]]
mls_h2_2025 = basic_info_h2.merge(flatten_goals_added(goals_added_h2_raw), on="player_id")

xpass_h2 = xpass_h2_raw[["player_id", "passes_completed_over_expected_p100", "avg_vertical_distance_yds"]]
mls_h2_2025 = mls_h2_2025.merge(xpass_h2, on="player_id", how="left")
mls_h2_2025 = mls_h2_2025.merge(players[["player_id", "player_name"]], on="player_id", how="left")

mls_h2_2025.to_parquet(PROCESSED_FOLDER / "asa_players_2025_h2.parquet")
print("Saved:", mls_h2_2025.shape)